# Create Marsden Fund Awards

Creates OpenAlex award rows from Royal Society Te Apārangi's official Marsden Fund awarded-grants workbooks.

**Data source:** `https://www.royalsociety.org.nz/what-we-do/funds-and-opportunities/marsden/awarded-grants`. The local exporter discovers the official 2008-2017 workbook and annual 2018-2025 announcement workbooks linked from the Royal Society Te Apārangi site.
**S3 location:** `s3a://openalex-ingest/awards/marsden/marsden_awards.parquet`
**OpenAlex funder:** `F4320335369` - Marsden Fund (no ROR/DOI in OpenAlex), country `NZ`.
**Provenance:** `marsden_awarded_grants`.
**Priority:** 222.

**Schema choices:**
- One row per unique Marsden `Project ID`; local full validation found 2,000 rows and 2,000 unique IDs, 2008-2025.
- Stable `funder_award_id = Project ID` (for example `25-AUT-013`), preserving the real citable Marsden grant identifier for WorkAwards matching.
- `amount` is parsed from `Funding (ex GST)` and `currency = NZD`; no amount waiver is needed.
- `display_name` uses the official project/media title, and `description` uses the official abstract when published.
- Investigator roles come from the workbook team sheets: first PI is `lead_investigator`, second PI (when present) is `co_lead_investigator`, and remaining PIs/AIs are stored in `investigators`.
- Affiliation country remains NULL because the workbooks publish organizations but no per-investigator country field. `start_year` is the award year; day-level dates are not published.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.marsden_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/marsden/marsden_awards.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_marsden_awards FROM openalex.awards.marsden_raw;


## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `project_id`, `display_name`, `description`, `media_title`, `amount`, `currency`, `funder_scheme`, `panel`, `category`, `funding_type`, `start_year`, `end_year`, `start_date`, `end_date`, `lead_given_name`, `lead_family_name`, `lead_affiliation_name`, `co_lead_given_name`, `co_lead_family_name`, `co_lead_affiliation_name`, `investigators_json`, `team_json`, `landing_page_url`, `source_page_url`, `source_workbook_url`, `provenance`, `funder_id`, `funder_display_name`, `downloaded_at`.


In [ ]:
%sql
DESCRIBE openalex.awards.marsden_raw;


In [ ]:
%sql
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'marsden_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|year|investigator|team|scheme|panel|category'
ORDER BY column_name;


In [ ]:
%sql
SELECT funder_award_id, display_name, lead_given_name, lead_family_name,
       lead_affiliation_name, amount, currency, funder_scheme, start_year
FROM openalex.awards.marsden_raw
LIMIT 10;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(lead_family_name) AS has_lead_family,
    COUNT(lead_affiliation_name) AS has_lead_affiliation,
    COUNT(co_lead_family_name) AS has_co_lead_family,
    COUNT(investigators_json) AS has_investigators_json,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS nzd_total,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.marsden_raw;


In [ ]:
%sql
SELECT panel, category, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS nzd_total
FROM openalex.awards.marsden_raw
GROUP BY panel, category
ORDER BY awards DESC
LIMIT 30;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS nzd_total
FROM openalex.awards.marsden_raw
GROUP BY start_year
ORDER BY TRY_CAST(start_year AS INT);


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS n
FROM openalex.awards.marsden_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY n DESC, funder_award_id
LIMIT 20;


## Step 1.6: Fail-Fast - Verify Marsden Funder Exists

F4320335369 is a Crossref-registered `F4320*` funder. This assertion prevents a silent zero-row transform if the funder dimension is unexpectedly missing.


In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'Expected exactly one Marsden Fund row for F4320335369') AS marsden_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320335369;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.marsden_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320335369 AS funder_id,
        COALESCE(MAX(display_name), 'Marsden Fund') AS display_name,
        MAX(ror_id) AS ror_id,
        MAX(doi) AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320335369
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        NULLIF(TRIM(display_name), '') AS display_name_norm,
        NULLIF(TRIM(description), '') AS description_norm,
        NULLIF(TRIM(lead_given_name), '') AS lead_given_name_norm,
        NULLIF(TRIM(lead_family_name), '') AS lead_family_name_norm,
        NULLIF(TRIM(lead_affiliation_name), '') AS lead_affiliation_name_norm,
        NULLIF(TRIM(co_lead_given_name), '') AS co_lead_given_name_norm,
        NULLIF(TRIM(co_lead_family_name), '') AS co_lead_family_name_norm,
        NULLIF(TRIM(co_lead_affiliation_name), '') AS co_lead_affiliation_name_norm,
        NULLIF(TRIM(funder_scheme), '') AS funder_scheme_norm,
        NULLIF(TRIM(funding_type), '') AS funding_type_norm,
        NULLIF(TRIM(landing_page_url), '') AS landing_page_url_norm,
        NULLIF(TRIM(investigators_json), '') AS investigators_json_norm
    FROM openalex.awards.marsden_raw
)
SELECT
    abs(xxhash64(CONCAT('4320335369:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name_norm AS display_name,
    rp.description_norm AS description,
    4320335369 AS funder_id,
    rp.funder_award_id,
    CASE WHEN rp.amount_double > 0 THEN rp.amount_double ELSE NULL END AS amount,
    CASE WHEN rp.amount_double > 0 THEN 'NZD' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    COALESCE(rp.funding_type_norm, 'research') AS funding_type,
    rp.funder_scheme_norm AS funder_scheme,
    'marsden_awarded_grants' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CASE
        WHEN rp.start_year_int > YEAR(current_date()) + 1 THEN NULL
        ELSE rp.start_year_int
    END AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN rp.lead_given_name_norm IS NOT NULL OR rp.lead_family_name_norm IS NOT NULL OR rp.lead_affiliation_name_norm IS NOT NULL THEN
            struct(
                rp.lead_given_name_norm AS given_name,
                rp.lead_family_name_norm AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.lead_affiliation_name_norm AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CASE
        WHEN rp.co_lead_given_name_norm IS NOT NULL OR rp.co_lead_family_name_norm IS NOT NULL OR rp.co_lead_affiliation_name_norm IS NOT NULL THEN
            struct(
                rp.co_lead_given_name_norm AS given_name,
                rp.co_lead_family_name_norm AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.co_lead_affiliation_name_norm AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >)
    END AS co_lead_investigator,
    CASE
        WHEN rp.investigators_json_norm IS NOT NULL THEN from_json(
            rp.investigators_json_norm,
            'ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>'
        )
        ELSE CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>)
    END AS investigators,
    rp.landing_page_url_norm AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320335369:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name_norm IS NOT NULL;


In [ ]:
%sql
SELECT COUNT(*) AS transformed_awards FROM openalex.awards.marsden_awards;


In [ ]:
%sql
SELECT id, COUNT(*) AS n
FROM openalex.awards.marsden_awards
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20;


In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, funder_scheme,
       start_year, lead_investigator.family_name AS lead_family,
       lead_investigator.affiliation.name AS lead_affiliation,
       size(investigators) AS investigator_count
FROM openalex.awards.marsden_awards
LIMIT 10;


## Step 3: Delete Existing Marsden Rows, Then Insert

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marsden_awarded_grants' AND priority = 222;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    222 AS priority
FROM openalex.awards.marsden_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marsden_awarded_grants' AND priority = 222;


In [ ]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(amount), 0) AS nzd_total,
    COUNT(description) AS has_description,
    COUNT(lead_investigator.family_name) AS has_lead_family,
    COUNT(co_lead_investigator.family_name) AS has_co_lead_family,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marsden_awarded_grants' AND priority = 222;


In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS nzd_total
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marsden_awarded_grants' AND priority = 222
GROUP BY funding_type, funder_scheme
ORDER BY awards DESC
LIMIT 30;


In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, funder_scheme,
       start_year, lead_investigator.family_name AS lead_family,
       lead_investigator.affiliation.name AS lead_affiliation,
       landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marsden_awarded_grants' AND priority = 222
ORDER BY funder_award_id
LIMIT 25;


In [ ]:
%sql
SELECT id, funder_award_id, works_api_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'marsden_awarded_grants' AND priority = 222
LIMIT 10;


## Handoff / Admin Notes

Contractor-complete only. The contractor has no S3 or Databricks access.

Admin follow-up:
- Upload `/tmp/marsden_awards.parquet` or rerun `scripts/local/marsden_to_s3.py --allow-shrink` with AWS credentials to write `s3://openalex-ingest/awards/marsden/marsden_awards.parquet`.
- Run this notebook on Databricks and inspect the Step 6 verification cells.
- After QA, update tracker status beyond Step 4 and only then consider scheduled job wiring.
